# RQ4 Lifestyle Risk Factors and Obesity

**Research question:** How do smoking and alcohol consumption relate to obesity classification?

This Kaggle notebook loads the raw dataset, generates the publication-ready table as CSV, saves the figure as PDF, and prints the evidence-based answer.

In [ ]:

# ============================================================
# Kaggle-ready setup: load raw obesity dataset and shared helpers
# ============================================================
import os, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import chi2_contingency, pearsonr, spearmanr, f_oneway

# Kaggle output directory
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Change this only if needed. On Kaggle, the dataset usually lives under /kaggle/input/...
DATASET_PATH = None


def find_dataset_file():
    """Find the obesity dataset on Kaggle or locally. Supports CSV and Excel."""
    if DATASET_PATH:
        return DATASET_PATH
    patterns = [
        '/kaggle/input/**/*.csv', '/kaggle/input/**/*.xlsx', '/kaggle/input/**/*.xls',
        './*.csv', './*.xlsx', './*.xls',
        '../input/**/*.csv', '../input/**/*.xlsx', '../input/**/*.xls'
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    # Prefer files whose name looks like the obesity dataset
    ranked = sorted(candidates, key=lambda p: ('obesity' not in os.path.basename(p).lower(), len(p)))
    if not ranked:
        raise FileNotFoundError('No CSV/XLSX dataset file found. Upload the raw dataset to Kaggle Input or set DATASET_PATH manually.')
    return ranked[0]


def load_dataset():
    path = find_dataset_file()
    print(f'Loading dataset from: {path}')
    if path.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
    return df


def clean_dataset(df):
    """Basic cleaning only; avoids changing the research meaning of variables."""
    df = df.copy()
    df = df.drop_duplicates().reset_index(drop=True)
    # Normalize object columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    return df


def add_obesity_group(df):
    """Collapse detailed target classes into paper-friendly groups."""
    df = df.copy()
    mapping = {
        'Insufficient_Weight': 'Underweight',
        'Normal_Weight': 'Normal',
        'Overweight_Level_I': 'Overweight',
        'Overweight_Level_II': 'Overweight',
        'Obesity_Type_I': 'Obese',
        'Obesity_Type_II': 'Obese',
        'Obesity_Type_III': 'Obese'
    }
    df['Obesity_Group'] = df['NObeyesdad'].map(mapping).fillna(df['NObeyesdad'])
    order = ['Underweight', 'Normal', 'Overweight', 'Obese']
    df['Obesity_Group'] = pd.Categorical(df['Obesity_Group'], categories=order, ordered=True)
    return df


def add_obesity_score(df):
    """Ordinal score for correlation/ANOVA summaries."""
    df = df.copy()
    score_map = {
        'Insufficient_Weight': 0,
        'Normal_Weight': 1,
        'Overweight_Level_I': 2,
        'Overweight_Level_II': 3,
        'Obesity_Type_I': 4,
        'Obesity_Type_II': 5,
        'Obesity_Type_III': 6
    }
    df['Obesity_Score'] = df['NObeyesdad'].map(score_map)
    return df


def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved table: {path}')
    return path


def save_figure(filename):
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, format='pdf', bbox_inches='tight')
    print(f'Saved figure: {path}')
    plt.show()
    return path


def percent_yes(series):
    return (series.astype(str).str.lower().eq('yes').mean() * 100)


def style_plot(title, xlabel=None, ylabel=None):
    plt.title(title, fontsize=14, weight='bold')
    if xlabel: plt.xlabel(xlabel, fontsize=11)
    if ylabel: plt.ylabel(ylabel, fontsize=11)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(axis='y', alpha=0.25)
    for spine in ['top', 'right']:
        plt.gca().spines[spine].set_visible(False)


df = add_obesity_score(add_obesity_group(clean_dataset(load_dataset())))
display(df.head())
print(df['NObeyesdad'].value_counts())


## Analysis, table, figure, and answer

In [ ]:

# RQ4: Smoking, alcohol consumption, and obesity classification
# Variables: SMOKE, CALC, NObeyesdad

def distribution_table(variable):
    ct = pd.crosstab(df[variable], df['Obesity_Group'])
    pct = ct.div(ct.sum(axis=1), axis=0).mul(100).reset_index().round(2)
    pct.insert(0, 'risk_factor', variable)
    pct = pct.rename(columns={variable: 'group'})
    pct['sample_size'] = ct.sum(axis=1).values
    return pct

smoke_tbl = distribution_table('SMOKE')
alcohol_tbl = distribution_table('CALC')
summary = pd.concat([smoke_tbl, alcohol_tbl], ignore_index=True)
save_table(summary, 'RQ4_smoking_alcohol_obesity_distribution.csv')
display(summary)

smoke_chi = chi2_contingency(pd.crosstab(df['SMOKE'], df['Obesity_Group']))
alcohol_chi = chi2_contingency(pd.crosstab(df['CALC'], df['Obesity_Group']))
tests = pd.DataFrame({
    'relationship': ['SMOKE vs obesity group', 'CALC vs obesity group'],
    'chi_square': [smoke_chi.statistic, alcohol_chi.statistic],
    'p_value': [smoke_chi.pvalue, alcohol_chi.pvalue]
}).round(5)
save_table(tests, 'RQ4_lifestyle_chi_square_tests.csv')
display(tests)

# Figure: obese percentage by smoking/alcohol group
fig_df = summary[['risk_factor', 'group', 'Obese', 'sample_size']].copy()
fig_df['label'] = fig_df['risk_factor'] + ': ' + fig_df['group'].astype(str)
plt.figure(figsize=(10, 5.5))
plt.bar(fig_df['label'], fig_df['Obese'])
plt.xticks(rotation=35, ha='right')
style_plot('Obese Percentage by Smoking and Alcohol Groups', 'Lifestyle risk group', 'Obese (%)')
save_figure('RQ4_smoking_alcohol_obesity.pdf')

print('\nANSWER TO RQ4')
max_row = fig_df.loc[fig_df['Obese'].idxmax()]
print(f"The lifestyle category with the highest obese percentage is: {max_row['label']} ({max_row['Obese']:.2f}%).")
print(f"SMOKE association p-value: {smoke_chi.pvalue:.4g}; CALC association p-value: {alcohol_chi.pvalue:.4g}.")
print('The table and chi-square tests show whether smoking and alcohol categories are statistically associated with obesity classification.')
